[실제 파이프라인]
<br>CT 슬라이스 → 2cls 모델 예측 (병변 후보 영역, binary) → 후보 영역을 crop → Qwen에 입력
                                                              ↓
                                          Qwen이 판단: (1) 진짜 병변인가? (2) 병변이면 어떤 클래스(EDH/IPH/IVH/SAH/SDH)인가?

* nnunet mbh-seg25 2cls 2d pp segmentation의 결과를 correcton 해보자
    * INPUT
        * pred: /home/jovyan/aicon-gamma-datavol-1/hjgoh/ich-vlm/nnUNet/nnUNet_predictions/Dataset002_MBHSeg25_1cls/2d_pp/MBH-Seg25
        * GT: /home/jovyan/aicon-gamma-datavol-1/hjgoh/ich-vlm/nnUNet/test_data_1cls/Test1_MBHSeg25_1cls
    * OUTPUT
        * file: /home/jovyan/aicon-gamma-datavol-1/hjgoh/ich-vlm/experiments/260818_vlm_mbhseg_dataset
        * slice 별 instance bbox overlay, cropped image / 이때, bbox 오버레이는 특정 인스턴스 하나만 오버레이
        * 5cls_dataset.json
        <br> bbox, id, class_id 등등 (reasoning 제외 )

* pipeline
    * semantic mask > instance mask or bbox
    * original + crop image
    * qwen input

* ground dataset
    * scan 별로
        * slice 별 2cls prediction / 5cls ground truth overlap
        * overlap 부분은 True / overlap 되지 않는 gt는 FN


---

In [16]:
import json
from pathlib import Path

import numpy as np
import nibabel as nib
import SimpleITK as sitk 
from PIL import Image, ImageDraw
from skimage.measure import label, regionprops
import pandas as pd

In [20]:
PRED_2CLS_DIR = Path("/home/jovyan/aicon-gamma-datavol-1/hjgoh/ich-vlm/experiments/260806_nnunet-infer/"
                      "mbhseg25_2cls/output/Dataset002_MBHSeg25_2cls/2d_pp/MBH-Seg25")
GT_5CLS_DIR = Path("/home/jovyan/aicon-gamma-datavol-1/hjgoh/ich-vlm/nnUNet/test_data/Test1_MBHSeg25/labelsTs")
IMAGES_DIR = Path("/home/jovyan/aicon-gamma-datavol-1/hjgoh/ich-vlm/nnUNet/test_data/Test1_MBHSeg25/imagesTs")
OUT_DIR = Path("/home/jovyan/aicon-gamma-datavol-1/hjgoh/ich-vlm/experiments/260818_vlm_mbhseg_dataset")
(OUT_DIR / "images").mkdir(parents=True, exist_ok=True)

In [6]:
def apply_window(slice_2d, center=40, width=80):
    """뇌 윈도우(HU 40±40 등)로 CT slice를 0~255 uint8로 변환. 기존 nnUNet_result.ipynb 함수와 동일 로직이면 그대로 import해서 써도 됨."""
    lo, hi = center - width / 2, center + width / 2
    clipped = np.clip(slice_2d, lo, hi)
    return ((clipped - lo) / (hi - lo) * 255).astype(np.uint8)

def load_lps_array(path):
    img_sitk = sitk.ReadImage(str(path))
    img_lps = sitk.DICOMOrient(img_sitk, "LPS")
    return sitk.GetArrayFromImage(img_lps)  # (z, y, x)

In [2]:
CLASS_NAMES = {
    1: "EDH", # (경막외출혈)
    2: "IPH", # (뇌실질출혈)
    3: "IVH", # (뇌실내출혈)
    4: "SAH", # (지주막하출혈)
    5: "SDH", # (경막하출혈)
}

In [7]:
def semantic_to_instance(mask: np.ndarray):
    """
    클래스별 connected component로 instance 분리. 
    class_id는 원본 semantic 값 그대로 유지
    (2cls mask에 쓰면 class_id는 항상 1).
    """
    instance_mask = np.zeros_like(mask, dtype=np.int32)
    regions_info = []
    next_id = 1
    for cls_id in np.unique(mask):
        if cls_id == 0:
            continue
        labeled = label(mask == cls_id)
        for region in regionprops(labeled):
            instance_mask[labeled == region.label] = next_id
            regions_info.append({
                "instance_id": next_id,
                "class_id": int(cls_id),
                "bbox": region.bbox,
                "area_px": int(region.area),
                "eccentricity": float(region.eccentricity),
            })
            next_id += 1
    return instance_mask, regions_info

2cls 예측 ↔ 5cls GT 매칭 

In [17]:
def match_pred_to_gt(pred_mask, pred_regions, gt_mask, gt_regions):
    """
    pred 인스턴스마다 GT(클래스 무관하게 합친 마스크)와 픽셀 겹침을 확인.
    - 안 겹치면: is_lesion=False, gt_class_id=None (FP)
    - 겹치면: is_lesion=True, 가장 많이 겹친 GT 인스턴스의 클래스를 gt_class_id로 기록
    fn_only: 어떤 pred와도 안 겹친 GT 인스턴스(통계용, 샘플엔 미포함)
    """
    gt_class_lookup = {r["instance_id"]: r["class_id"] for r in gt_regions}

    matched = []
    matched_gt_ids = set()

    for pred in pred_regions:
        pred_pixels = pred_mask == pred["instance_id"]
        overlap_ids = np.unique(gt_mask[pred_pixels])
        overlap_ids = overlap_ids[overlap_ids > 0]

        if len(overlap_ids) == 0:
            matched.append({**pred, "is_lesion": False, "gt_class_id": None})
            continue

        overlap_counts = {gid: int(np.sum((gt_mask == gid) & pred_pixels)) for gid in overlap_ids}
        best_gt_id = max(overlap_counts, key=overlap_counts.get)
        matched_gt_ids.add(best_gt_id)
        matched.append({**pred, "is_lesion": True, "gt_class_id": gt_class_lookup[best_gt_id]})

    fn_only = [r for r in gt_regions if r["instance_id"] not in matched_gt_ids]
    return matched, fn_only

In [18]:
def draw_single_bbox_overlay(image_2d, bbox, color=(255, 0, 0)):
    """이 인스턴스의 pred bbox 하나만 그린다 — GT는 절대 표시하지 않음."""
    rgb = np.stack([image_2d] * 3, axis=-1).astype(np.uint8)
    pil_img = Image.fromarray(rgb)
    draw = ImageDraw.Draw(pil_img)
    min_row, min_col, max_row, max_col = bbox
    draw.rectangle([min_col, min_row, max_col, max_row], outline=color, width=2)
    return pil_img


def crop_zoomed(image_2d, bbox, margin=10):
    min_row, min_col, max_row, max_col = bbox
    h, w = image_2d.shape
    r0, r1 = max(0, min_row - margin), min(h, max_row + margin)
    c0, c1 = max(0, min_col - margin), min(w, max_col + margin)
    return Image.fromarray(image_2d[r0:r1, c0:c1].astype(np.uint8))

In [ ]:
def build_slice_samples(case_id, slice_idx, image_2d, pred_2d, gt_2d, out_dir):
    windowed = apply_window(image_2d)

    pred_mask, pred_regions = semantic_to_instance(pred_2d)
    gt_mask, gt_regions = semantic_to_instance(gt_2d)

    if not pred_regions:
        return [], len(gt_regions)

    matched, fn_only = match_pred_to_gt(pred_mask, pred_regions, gt_mask, gt_regions)

    samples = []
    for region in matched:
        overlay_img = draw_single_bbox_overlay(windowed, region["bbox"])
        overlay_path = out_dir / "images" / f"{case_id}_s{slice_idx:03d}_inst{region['instance_id']}_overlay.png"
        overlay_img.save(overlay_path)

        crop_img = crop_zoomed(windowed, region["bbox"])
        crop_path = out_dir / "images" / f"{case_id}_s{slice_idx:03d}_inst{region['instance_id']}_crop.png"
        crop_img.save(crop_path)

        gt_class_id = region["gt_class_id"]

        samples.append({
            "id": f"{case_id}_s{slice_idx:03d}_inst{region['instance_id']}",
            "case_id": case_id,
            "slice_idx": slice_idx,
            "bbox": list(region["bbox"]),
            "area_px": region["area_px"],
            "is_lesion": region["is_lesion"],
            "gt_class_id": gt_class_id,
            "gt_class_name": CLASS_NAMES[gt_class_id] if gt_class_id else None,
            "images": {
                "overlay": str(overlay_path),
                "cropped": str(crop_path),
            },
        })

    return samples, len(fn_only)

In [21]:
all_samples = []
fn_log = []

for pred_path in sorted(PRED_2CLS_DIR.glob("*.nii.gz")):
    case_id = pred_path.stem.replace(".nii", "")
    gt_path = GT_5CLS_DIR / pred_path.name
    img_path = IMAGES_DIR / f"{case_id}_0000.nii.gz"

    if not gt_path.exists() or not img_path.exists():
        print(f"[건너뜀] {case_id}: gt 또는 원본 이미지 없음")
        continue

    pred_vol = load_lps_array(pred_path).astype(np.uint8)
    gt_vol = load_lps_array(gt_path).astype(np.uint8)
    img_vol = load_lps_array(img_path)

    for z in range(pred_vol.shape[0]):
        pred_2d, gt_2d, img_2d = pred_vol[z], gt_vol[z], img_vol[z]
        if pred_2d.sum() == 0 and gt_2d.sum() == 0:
            continue

        samples, n_fn = build_slice_samples(case_id, z, img_2d, pred_2d, gt_2d, OUT_DIR)
        all_samples.extend(samples)
        if n_fn > 0:
            fn_log.append({"case_id": case_id, "slice_idx": z, "n_fn": n_fn})

print(f"총 샘플 수: {len(all_samples)}, FN 발생 슬라이스 수: {len(fn_log)}")

총 샘플 수: 958, FN 발생 슬라이스 수: 235


In [22]:
with open(OUT_DIR / "dataset.json", "w", encoding="utf-8") as f:
    json.dump(all_samples, f, ensure_ascii=False, indent=2)

pd.DataFrame(fn_log).to_csv(OUT_DIR / "fn_log.csv", index=False)

print(f"저장 완료: {OUT_DIR / 'dataset.json'} ({len(all_samples)}개 샘플)")
print(f"FN 로그: {OUT_DIR / 'fn_log.csv'}")

저장 완료: /home/jovyan/aicon-gamma-datavol-1/hjgoh/ich-vlm/experiments/260818_vlm_mbhseg_dataset/dataset.json (958개 샘플)
FN 로그: /home/jovyan/aicon-gamma-datavol-1/hjgoh/ich-vlm/experiments/260818_vlm_mbhseg_dataset/fn_log.csv


---

* nnunet mbh-seg25 5cls 2d pp segmentation의 결과를 correcton 해보자
    * INPUT
        * pred: /home/jovyan/aicon-gamma-datavol-1/hjgoh/ich-vlm/nnUNet/nnUNet_predictions/Dataset001_MBHSeg25/2d_pp/MBH-Seg25
        * GT: /home/jovyan/aicon-gamma-datavol-1/hjgoh/ich-vlm/nnUNet/test_data/Test1_MBHSeg25
    * OUTPUT
        * file: /home/jovyan/aicon-gamma-datavol-1/hjgoh/ich-vlm/experiments/260818_vlm_mbhseg_5cls_dataset
        * slice 별 instance bbox overlay, cropped image / 이때, bbox 오버레이는 특정 인스턴스 하나만 오버레이
        * 5cls_dataset.json
        <br> bbox, id, class_id 등등 (reasoning 제외 )

In [4]:
PRED_5CLS_DIR = Path("/home/jovyan/aicon-gamma-datavol-1/hjgoh/ich-vlm/nnUNet/nnUNet_predictions/Dataset001_MBHSeg25/2d_pp/MBH-Seg25")
GT_5CLS_DIR = Path("/home/jovyan/aicon-gamma-datavol-1/hjgoh/ich-vlm/nnUNet/test_data/Test1_MBHSeg25/labelsTs")
IMAGES_DIR = Path("/home/jovyan/aicon-gamma-datavol-1/hjgoh/ich-vlm/nnUNet/test_data/Test1_MBHSeg25/imagesTs")
OUT_DIR = Path("/home/jovyan/aicon-gamma-datavol-1/hjgoh/ich-vlm/experiments/260818_vlm_mbhseg_5cls_dataset")
(OUT_DIR / "images").mkdir(parents=True, exist_ok=True)

5cls 예측 ↔ 5cls GT 매칭 

In [8]:
def match_pred_to_gt(pred_mask, pred_regions, gt_mask, gt_regions):
    """
    pred 인스턴스마다 GT(클래스 무관하게 합친 마스크)와 픽셀 겹침을 확인.
    - 안 겹치면: is_lesion=False, gt_class_id=None (FP)
    - 겹치면: is_lesion=True, 가장 많이 겹친 GT 인스턴스의 클래스를 gt_class_id로 기록
    fn_only: 어떤 pred와도 안 겹친 GT 인스턴스(통계용, 샘플엔 미포함)
    """
    gt_class_lookup = {r["instance_id"]: r["class_id"] for r in gt_regions}

    matched = []
    matched_gt_ids = set()

    for pred in pred_regions:
        pred_pixels = pred_mask == pred["instance_id"]
        overlap_ids = np.unique(gt_mask[pred_pixels])
        overlap_ids = overlap_ids[overlap_ids > 0]

        if len(overlap_ids) == 0:
            matched.append({**pred, "is_lesion": False, "gt_class_id": None})
            continue

        overlap_counts = {gid: int(np.sum((gt_mask == gid) & pred_pixels)) for gid in overlap_ids}
        best_gt_id = max(overlap_counts, key=overlap_counts.get)
        matched_gt_ids.add(best_gt_id)
        matched.append({**pred, "is_lesion": True, "gt_class_id": gt_class_lookup[best_gt_id]})

    fn_only = [r for r in gt_regions if r["instance_id"] not in matched_gt_ids]
    return matched, fn_only


In [9]:
def draw_single_bbox_overlay(image_2d, bbox, color=(255, 0, 0)):
    """이 인스턴스의 pred bbox 하나만 그린다 — GT는 절대 표시하지 않음."""
    rgb = np.stack([image_2d] * 3, axis=-1).astype(np.uint8)
    pil_img = Image.fromarray(rgb)
    draw = ImageDraw.Draw(pil_img)
    min_row, min_col, max_row, max_col = bbox
    draw.rectangle([min_col, min_row, max_col, max_row], outline=color, width=2)
    return pil_img


def crop_zoomed(image_2d, bbox, margin=10):
    min_row, min_col, max_row, max_col = bbox
    h, w = image_2d.shape
    r0, r1 = max(0, min_row - margin), min(h, max_row + margin)
    c0, c1 = max(0, min_col - margin), min(w, max_col + margin)
    return Image.fromarray(image_2d[r0:r1, c0:c1].astype(np.uint8))

In [ ]:
def build_slice_samples(case_id, slice_idx, image_2d, pred_2d, gt_2d, out_dir):
    windowed = apply_window(image_2d)

    pred_mask, pred_regions = semantic_to_instance(pred_2d)
    gt_mask, gt_regions = semantic_to_instance(gt_2d)

    if not pred_regions:
        return [], len(gt_regions)

    matched, fn_only = match_pred_to_gt(pred_mask, pred_regions, gt_mask, gt_regions)

    samples = []
    for region in matched:
        overlay_img = draw_single_bbox_overlay(windowed, region["bbox"])
        overlay_path = out_dir / "images" / f"{case_id}_s{slice_idx:03d}_inst{region['instance_id']}_overlay.png"
        overlay_img.save(overlay_path)

        crop_img = crop_zoomed(windowed, region["bbox"])
        crop_path = out_dir / "images" / f"{case_id}_s{slice_idx:03d}_inst{region['instance_id']}_crop.png"
        crop_img.save(crop_path)

        pred_class_id = region["class_id"]
        gt_class_id = region["gt_class_id"]

        samples.append({
            "id": f"{case_id}_s{slice_idx:03d}_inst{region['instance_id']}",
            "case_id": case_id,
            "slice_idx": slice_idx,
            "bbox": list(region["bbox"]),
            "area_px": region["area_px"],
            "pred_class_id": pred_class_id,
            "pred_class_name": CLASS_NAMES[pred_class_id],
            "is_lesion": region["is_lesion"],
            "gt_class_id": gt_class_id,
            "gt_class_name": CLASS_NAMES[gt_class_id] if gt_class_id else None,
            "class_correct": (pred_class_id == gt_class_id) if region["is_lesion"] else None,
            "images": {
                "overlay": str(overlay_path),
                "cropped": str(crop_path),
            },
        })

    return samples, len(fn_only)

In [12]:
# main loop
all_samples = []
fn_log = []

pred_cases = sorted(p.stem.replace(".nii", "") for p in PRED_5CLS_DIR.glob("*.nii.gz"))
print(f"전체 케이스 수: {len(pred_cases)}")

for case_id in pred_cases:
    pred_path = PRED_5CLS_DIR / f"{case_id}.nii.gz"
    gt_path = GT_5CLS_DIR / f"{case_id}.nii.gz"
    img_path = IMAGES_DIR / f"{case_id}_0000.nii.gz"

    if not gt_path.exists() or not img_path.exists():
        print(f"[건너뜀] {case_id}: gt 또는 원본 이미지 없음")
        continue

    pred_vol = load_lps_array(pred_path).astype(np.uint8)
    gt_vol = load_lps_array(gt_path).astype(np.uint8)
    img_vol = load_lps_array(img_path)

    for z in range(pred_vol.shape[0]):
        pred_2d, gt_2d, img_2d = pred_vol[z], gt_vol[z], img_vol[z]
        if pred_2d.sum() == 0 and gt_2d.sum() == 0:
            continue

        samples, n_fn = build_slice_samples(case_id, z, img_2d, pred_2d, gt_2d, OUT_DIR)
        all_samples.extend(samples)
        if n_fn > 0:
            fn_log.append({"case_id": case_id, "slice_idx": z, "n_fn": n_fn})

print(f"총 샘플 수: {len(all_samples)}, FN 발생 슬라이스 수: {len(fn_log)}")


전체 케이스 수: 48
총 샘플 수: 984, FN 발생 슬라이스 수: 236


In [15]:
with open(OUT_DIR / "5cls_dataset.json", "w", encoding="utf-8") as f:
    json.dump(all_samples, f, ensure_ascii=False, indent=2)

pd.DataFrame(fn_log).to_csv(OUT_DIR / "fn_log.csv", index=False)

print(f"저장 완료: {OUT_DIR / '5cls_dataset.json'} ({len(all_samples)}개 샘플)")
print(f"FN 로그: {OUT_DIR / 'fn_log.csv'}")

저장 완료: /home/jovyan/aicon-gamma-datavol-1/hjgoh/ich-vlm/experiments/260818_vlm_mbhseg_5cls_dataset/5cls_dataset.json (984개 샘플)
FN 로그: /home/jovyan/aicon-gamma-datavol-1/hjgoh/ich-vlm/experiments/260818_vlm_mbhseg_5cls_dataset/fn_log.csv
